In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Define paths
data_dir = "../data/1000images"
img_height = 224
img_width = 224
batch_size = 32

# 1. Load the dataset and split into Training (80%) and Validation (20%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# 2. Data Augmentation (Crucial for the "Robustness" score) 
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal_and_vertical"),
  layers.RandomRotation(0.2),
])

# 3. Rescaling (Normalizing pixel values to [0,1]) 
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

Found 997 files belonging to 39 classes.
Using 798 files for training.
Found 997 files belonging to 39 classes.
Using 199 files for validation.


In [3]:
from tensorflow.keras.applications import MobileNetV2

# 1. Load the pre-trained MobileNetV2 model (without the top classification layer)
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # Freeze the pre-trained weights

# 2. Add custom layers for your 39 eye disease categories
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),  # Helps prevent overfitting
    layers.Dense(39, activation='softmax') # 39 categories [cite: 30]
])

# 3. Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 4. Train the model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10  # You can increase this later for better accuracy
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 7s 1us/step
Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 26s 677ms/step - accuracy: 0.1792 - loss: 3.2481 - val_accuracy: 0.2362 - val_loss: 3.0275
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 17s 646ms/step - accuracy: 0.3446 - loss: 2.5209 - val_accuracy: 0.3668 - val_loss: 2.5426
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 18s 695ms/step - accuracy: 0.4173 - loss: 2.1783 - val_accuracy: 0.4623 - val_loss: 2.2243
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 18s 698ms/step - accuracy: 0.4749 - loss: 1.8977 - val_accuracy: 0.4121 - val_loss: 2.1897
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 16s 608ms/step - accuracy: 0.5063 - loss: 1.7154 - val_accuracy: 0.4070 - val_loss: 2.0549
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 19s 750ms/step - accuracy: 0.5564 - loss: 1.6511 - val_accuracy: 0.4975 - val_loss: 1.8482
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 20s 748ms/step - accuracy: 0.5739 - loss: 1.4632 - val_accuracy: 0.4724 - val_loss: 1.8675
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 18s 713ms/

In [5]:
import os
import json

# Create the directory if it doesn't exist
os.makedirs('../backend/models', exist_ok=True)

# 1. Save the model
model.save('../backend/models/eye_disease_model.keras')

# 2. FIX: Get class names from the source directory
# We go back to the original data_dir to get the folder names
data_dir = "../data/1000images"
class_names = sorted([f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))])

with open('../backend/models/class_names.json', 'w') as f:
    json.dump(class_names, f)

print(f"✅ Model saved and {len(class_names)} classes documented!")

✅ Model saved and 39 classes documented!
